In [1]:
!pip install transformers==4.43.0, accelerate==0.30.0, torch==2.3.0, torchvision==0.18.0, bitsandbytes
!pip install tqdm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 90.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.4/302.4 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 779.1/779.1 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 87.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124

In [2]:
import json
import os
import re
import random
import datetime
import traceback
import torch
import requests
from PIL import Image
import torch
from tqdm.auto import tqdm
from transformers import AutoConfig, AutoModelForCausalLM, AutoProcessor, BitsAndBytesConfig
from tqdm.auto import tqdm


class PhiVQAEvaluator:
    

    def __init__(self, config_path):
        with open(config_path) as f:
            self.config = json.load(f)

        # Get Phi-specific configuration
        self.model_config = self.config["open_source_models"]["phi"]
        self.sampling_percentage = self.config.get("sampling_percentage", 100)
        self.unable_to_respond_aware = self.config.get("unable_to_respond_aware", True)
        self.initialize_model()

    def _create_prompt(self, question, NLP_tag_text, entities=None, entity_types=None):
        
        annotated_question = self.annotate_question_with_entities(question, entities or [], entity_types or [])
    
        unable_to_respond_line = "- If you are not certain or the context does not match, return 'Unable to determine'."
        prompt = (
            "You are a highly precise AI assistant for document analysis."
            "Your task is to answer questions by meticulously checking the provided document content.\n\n"
            "The document text is structured with OCR text where entities are annotated using NLP tags.\n\n"
            "--- DOCUMENT CONTENT ---\n"
            f"{NLP_tag_text}\n"
            "--- END OF DOCUMENT CONTENT ---\n\n"
            "Guidelines:\n"
            "- Provide a concise answer (a single word or short phrase).\n"
            "- Your answer MUST be based on the information provided above.\n"
            "- **Crucially, verify the context. An entity (e.g., a year, a name) found in a different section or page than the one implied by the question makes the question unanswerable from that context.**\n"
            "- For example, if the question asks about a main fact, an answer found only in an [Endnote] or on a different page is NOT valid.\n"
            f"{unable_to_respond_line}\n\n"
            f"Question: {annotated_question}"
        )
        return prompt


    def initialize_model(self):
        print("Initializing Phi model...")
        print("Model configuration:", self.model_config)
        model_name = self.model_config["model_name"]

        # 1. Force FP16 precision (Massive GPU speedup)
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        torch_dtype = torch.float16 if device.type == "cuda" else torch.float32
                
        # 2. Load the config
        config = AutoConfig.from_pretrained(model_name, trust_remote_code=True)

        # 3. Load the model NATIVELY without bitsandbytes
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            config=config,
            trust_remote_code=True,
            torch_dtype=torch_dtype,  
            cache_dir="/data1/hf_cache/models",
            device_map="auto", 
            attn_implementation="eager" # PyTorch optimized attention
        )

        self.processor = AutoProcessor.from_pretrained(
            model_name, 
            trust_remote_code=True,
        )

        self.max_tokens = self.model_config.get("max_tokens", 1024)
        print(f"Phi model initialized successfully.")


    def _cleanup_model(self):
        if hasattr(self, "model"):
            del self.model
            torch.cuda.empty_cache()
            import gc

            gc.collect()

    def get_structured_ocr_text(self, pages_layout):
        full_structured_text = []
        for page_id, page_data in pages_layout.items():
            image_filename = os.path.basename(page_id)
            full_structured_text.append(f"--- Page: {image_filename} ---")
            layout_objects = page_data.get("layout_analysis", {})
            sorted_objects = sorted(
                layout_objects.values(),
                key=lambda obj: (obj.get("BBOX", [0, 0])[1], obj.get("BBOX", [0, 0])[0])
            )
            for obj in sorted_objects:
                obj_type = obj.get("ObjectType", "unknown").replace("_", " ").title()
                ocr_text = obj.get("OCR", "").strip()
                if ocr_text:
                    full_structured_text.append(f"[{obj_type}]: {ocr_text}")
        return "\n".join(full_structured_text)


    def get_sorted_ocr_text(self, layout_analysis: dict, patch_entities: dict = None) -> str:
        """
        Restituisce OCR ordinato per y,x (come prima) ma con entità NLP annotate.
        patch_entities: mappa obj_id -> { ..., "entities": [ {start,end,text,label}, ... ] }
        La funzione usa start/end se validi; altrimenti cerca il testo come fallback.
        """
        ocr_items = []
        for obj_id, obj in layout_analysis.items():
            if isinstance(obj, dict) and "OCR" in obj and "BBOX" in obj:
                bbox = obj["BBOX"]
                ocr_items.append((bbox[1], bbox[0], obj_id, obj["OCR"]))
        ocr_items.sort()

        annotated_lines = []
        for _, _, obj_id, ocr_text in ocr_items:
            text = ocr_text or ""
            entities = []
            if patch_entities:
                # patch_entities per pagina fornisce key object0, object1, ...
                entities = patch_entities.get(obj_id, {}).get("entities", []) or []

            if entities:
                # processa dall'ultima entità (indice maggiore) alla prima
                for ent in sorted(entities, key=lambda e: e.get("start", 0), reverse=True):
                    ent_text = ent.get("text", "")
                    start = ent.get("start")
                    end = ent.get("end")
                    tag = ent.get("label", "entity").lower()

                    applied = False
                    # 1) se start/end sono validi e corrispondono, usa quelli
                    if isinstance(start, int) and isinstance(end, int) and 0 <= start < end <= len(text):
                        substr = text[start:end]
                        # controllo di coerenza (togli spazi estremi)
                        if substr.strip().lower() == ent_text.strip().lower() or ent_text.strip().lower() in substr.strip().lower():
                            text = text[:start] + f"<{tag}>{text[start:end]}</{tag}>" + text[end:]
                            applied = True

                    # 2) fallback: cerca il testo (case-insensitive) e tagga la prima occorrenza
                    if not applied and ent_text:
                        idx = text.lower().find(ent_text.strip().lower())
                        if idx != -1:
                            length = len(ent_text.strip())
                            text = text[:idx] + f"<{tag}>{text[idx:idx+length]}</{tag}>" + text[idx+length:]
                            applied = True

                    # se non è stato applicato, saltiamo (evita crash)
                    # potresti registrare debug qui se vuoi sapere cosa non è stato taggato

            annotated_lines.append(text.strip())

        return "\n".join(annotated_lines)



    def annotate_question_with_entities(self, question: str, entities: list, entity_types: list) -> str:
        """
        Inserisce tag <entity_type>... </entity_type> nella domanda.
        - entities: lista di dict (o stringhe). Se è una lista di dict usa 'text'.
        - entity_types: lista parallela di tipi (es. "year_numerical_value").
        Sostituisce la prima occorrenza di ogni entità nella domanda (case-insensitive),
        preservando la capitalizzazione originale.
        """
        if not question:
            return question

        pairs = []
        # Prima prova a usare entity_types (zip)
        if entity_types and entities:
            for ent, etype in zip(entities, entity_types):
                text = ent.get("text", "") if isinstance(ent, dict) else str(ent)
                if text:
                    pairs.append((text, etype))
        else:
            # Fallback: estrai label dall'entità se presente
            for ent in entities or []:
                if isinstance(ent, dict):
                    text = ent.get("text", "")
                    tag = ent.get("label", "entity")
                else:
                    text = str(ent)
                    tag = "entity"
                if text:
                    pairs.append((text, tag))

        # Ordina per lunghezza del testo decrescente per evitare match parziali
        pairs = sorted(pairs, key=lambda x: len(x[0]), reverse=True)

        out = question
        for text, tag in pairs:
            if not text:
                continue
            pattern = re.compile(re.escape(text), flags=re.IGNORECASE)
            def _repl(m):
                return f"<{tag.lower()}>{m.group(0)}</{tag.lower()}>"
            out, n = pattern.subn(_repl, out, count=1)  # sostituisci solo la prima occorrenza
        return out


    def generate_answer(self, question, image_paths, NLP_tag_dict, entities=None, entity_types=None):
        try:
            window_size = self.model_config.get("batch_size", 1)
            if window_size > 1:
                stride = self.model_config.get("stride", window_size // 2)
            else:
                stride = 1
                
            total_images = len(image_paths)
            total_windows = max(1, (total_images - window_size + stride) // stride)
            all_responses = []

            device = next(self.model.parameters()).device

            for window_idx in range(total_windows):
                start_idx = window_idx * stride
                end_idx = min(start_idx + window_size, total_images)
                window_paths = image_paths[start_idx:end_idx]

                if window_idx == total_windows - 1 and end_idx < total_images:
                    window_paths = image_paths[-window_size:]

                # Extract only the NLP text for this specific window
                window_nlp_text = ""
                if NLP_tag_dict:
                    texts = [NLP_tag_dict.get(path, "") for path in window_paths]
                    window_nlp_text = "\n\n".join(filter(None, texts))
                
                # Create prompt ONCE
                question_prompt = self._create_prompt(question, window_nlp_text, entities, entity_types)

                user_prompt = "<|user|>\n"
                assistant_prompt = "<|assistant|>\n"
                prompt_suffix = "<|end|>\n"

                # Prepare the Phi-3 specific multi-image prompt
                if len(window_paths) == 1:
                    prompt = f"{user_prompt}<|image_1|>\n {question_prompt}{prompt_suffix}{assistant_prompt}"
                    images = Image.open(window_paths[0])
                elif len(window_paths) == 2:
                    prompt = f"{user_prompt}<|image_1|>\n<|image_2|>\n {question_prompt}{prompt_suffix}{assistant_prompt}"
                    images = [Image.open(path) for path in window_paths]
                else:
                    prompt = f"{user_prompt}<|image_1|>\n<|image_2|>\n<|image_3|>\n {question_prompt}{prompt_suffix}{assistant_prompt}"
                    images = [Image.open(path) for path in window_paths]

                with torch.inference_mode():
                    inputs = self.processor(
                        prompt,
                        images,
                        return_tensors="pt",
                    ).to(device)

                    generate_ids = self.model.generate(
                        **inputs,
                        max_new_tokens=self.max_tokens,
                        eos_token_id=self.processor.tokenizer.eos_token_id,
                    )
                    
                    generate_ids = generate_ids[:, inputs["input_ids"].shape[1] :]
                    response = self.processor.batch_decode(
                        generate_ids,
                        skip_special_tokens=True,
                        clean_up_tokenization_spaces=False,
                    )[0].strip()

                all_responses.append(
                    {
                        "pages": window_paths,  
                        "answer": response,
                    }
                )
                
                del inputs
                del generate_ids
                torch.cuda.empty_cache()

            return {
                "answer": all_responses,
                "query": question,
                "image_paths": image_paths,
                "analysis_type": f"window_size_{window_size}",
            }

        except Exception as e:
            print(f"Error in generate_answer: {str(e)}")
            print(f"Full error: {traceback.format_exc()}")
            return {
                "answer": "Unable to determine: error",
                "error": str(e),
                "traceback": traceback.format_exc(),
            }

    def _save_results(self, data):
        output_file = self.model_config["name"]+"_"+self.config["output_file"]
        if self.config["ocr_enabled"] and not self.config["unable_to_respond_aware"]:
            output_file = output_file.replace(".json", "_OCR_UNABLE.json")
        elif self.config["ocr_enabled"]:
            output_file = output_file.replace(".json", "_OCR.json")
        elif not self.config["unable_to_respond_aware"]:
            output_file = output_file.replace(".json", "_UNABLE.json")
        try:
            with open(output_file, "w") as f:
                json.dump(data, f, indent=2)
            print(f"Results successfully saved to {output_file}")
        except Exception as e:
            print(f"Error saving results: {str(e)}")



    def evaluate(self):
        try:
            # 1. Determine the output file name early
            output_file = self.model_config["name"]+"_"+self.config["output_file"]
            if self.config["ocr_enabled"] and not self.config["unable_to_respond_aware"]:
                output_file = output_file.replace(".json", "_OCR_UNABLE.json")
            elif self.config["ocr_enabled"]:
                output_file = output_file.replace(".json", "_OCR.json")
            elif not self.config["unable_to_respond_aware"]:
                output_file = output_file.replace(".json", "_UNABLE.json")

            # 2. Check if we have an existing save file to resume from
            if os.path.exists(output_file):
                with open(output_file) as f:
                    data = json.load(f)
                print(f"Resuming from existing save file: {output_file}")
            else:
                with open(self.config["input_file"]) as f:
                    data = json.load(f)
                print(f"Successfully loaded input file: {self.config['input_file']}")

            # Sample questions
            total_questions = len(data["corrupted_questions"])
            num_samples = int(total_questions * (self.sampling_percentage / 100))

            if self.sampling_percentage < 100:
                sampled_questions = random.sample(
                    data["corrupted_questions"], num_samples
                )
                data["corrupted_questions"] = sampled_questions
                print(f"Sampled {num_samples} questions ({self.sampling_percentage}%) for evaluation")
            else:
                print("Processing 100% of questions (no sampling)")

            processed_count = 0
            success_count = 0
            error_count = 0
            skipped_count = 0

            for item in tqdm(data["corrupted_questions"]):
                
                # 3. CRITICAL FIX: Skip if we already answered this question in a previous run
                if "verification_result" in item and "vqa_results" in item["verification_result"] and len(item["verification_result"]["vqa_results"]) > 0:
                    skipped_count += 1
                    continue

                try:
                    processed_count += 1
                    item.setdefault("verification_result", {}).setdefault("vqa_results", [])
                    question = item["corrupted_question"]
                    entities = item.get("corrupted_entities", [])
                    entity_types = item.get("entity_type", [])

                    pages = item["layout_analysis"]["pages"]
                    patch_entities = item.get("patch_entities", {})

                    NLP_tag_dict = {}
                    image_paths = []
                    
                    for page_id, page_data in pages.items():
                        image_filename = os.path.basename(page_id)
                        image_path = os.path.join(self.config["images_base_path"], image_filename)
                        image_paths.append(image_path)
                        
                        layout = page_data.get("layout_analysis", {})
                        patches = patch_entities.get(page_id, {})
                        annotated_ocr = self.get_sorted_ocr_text(layout, patches)
                        
                        page_text = f"--- Page: {image_filename} ---\n{annotated_ocr}"
                        NLP_tag_dict[image_path] = page_text

                    # Now we are passing a dictionary, which fixes the .get() error
                    result = self.generate_answer(question, image_paths, NLP_tag_dict, entities, entity_types)

                    vqa_result = {
                        "model_type": "Phi",
                        "model_config": {
                            "batch_size": self.model_config.get("batch_size", 1),
                            "max_tokens": self.max_tokens,
                        },
                        "ocr_enabled": bool(NLP_tag_dict),
                        "question": question,
                        "answer": result.get("answer", "Unable to determine"),
                        "image_paths": image_paths,
                        "analysis_type": result.get("analysis_type", ""),
                        "timestamp": datetime.datetime.now().isoformat(),
                        #"NLP_tag_text": NLP_tag_text
                    }

                    if "error" in result:
                        vqa_result["error"] = result["error"]
                        vqa_result["traceback"] = result.get("traceback", "")
                        error_count += 1
                    else:
                        success_count += 1

                    item["verification_result"]["vqa_results"].append(vqa_result)

                    # 4. CRITICAL FIX: Uncomment the intermediate save
                    if processed_count % 10 == 0:
                        # Pass the specific output_file so we overwrite the correct resume file
                        with open(output_file, "w") as f:
                            json.dump(data, f, indent=2)
                        # print("Intermediate results saved")

                except Exception as e:
                    print(f"Error processing question: {str(e)}")
                    print(f"Full error: {traceback.format_exc()}")
                    error_count += 1

            # Final statistics
            print(f"\nProcessing completed:")
            print(f"Total questions skipped (already done): {skipped_count}")
            print(f"Total questions newly processed: {processed_count}")
            print(f"Successful generations: {success_count}")
            print(f"Errors encountered: {error_count}")
            if processed_count > 0:
                print(f"Success rate: {(success_count/processed_count)*100:.2f}%")

            # Save final results
            with open(output_file, "w") as f:
                json.dump(data, f, indent=2)
            print(f"Final results saved to {output_file}")

        except Exception as e:
            print(f"Critical error in evaluate: {str(e)}")
            print(f"Full error: {traceback.format_exc()}")
        finally:
            self._cleanup_model()

def main():
    config_path = "/kaggle/input/datasets/matteopetrelli/config-phi/config_kaggle.json"  # Update this path to match your config file location
    evaluator = PhiVQAEvaluator(config_path)
    evaluator.evaluate()

if __name__ == "__main__":
    main()


2026-05-06 15:59:26.138597: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778083166.514277      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778083166.629751      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778083167.590815      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778083167.590862      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778083167.590864      23 computation_placer.cc:177] computation placer alr

Initializing Phi model...
Model configuration: {'name': 'Phi', 'model_name': 'microsoft/Phi-3.5-vision-instruct', 'batch_size': 1, 'max_tokens': 1024}


config.json: 0.00B [00:00, ?B/s]

configuration_phi3_v.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3.5-vision-instruct:
- configuration_phi3_v.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

Phi model initialized successfully.
Successfully loaded input file: /kaggle/input/datasets/matteopetrelli/dude-questions/DUDE_fixed.json
Processing 100% of questions (no sampling)


  0%|          | 0/187 [00:00<?, ?it/s]

The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.
You are not running the flash-attention implementation, expect numerical differences.


Error in generate_answer: CUDA out of memory. Tried to allocate 5.32 GiB. GPU 
Full error: Traceback (most recent call last):
  File "/tmp/ipykernel_23/3757324315.py", line 262, in generate_answer
    generate_ids = self.model.generate(
                   ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/_contextlib.py", line 115, in decorate_context
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py", line 1989, in generate
    result = self._sample(
             ^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py", line 2932, in _sample
    outputs = self(**model_inputs, return_dict=True)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1532, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^